In [1]:
!git clone -q https://github.com/duakhurram2212/smart_retail.git
%cd smart_retail
!pip install -q xgboost shap scikit-learn scipy pyarrow

/content/smart_retail


In [2]:
from google.colab import files
import os, shutil
os.makedirs("data/raw", exist_ok=True)
files.upload()
shutil.move("holidays_events.csv", "data/raw/holidays_events.csv")

Saving holidays_events.csv to holidays_events.csv


'data/raw/holidays_events.csv'

In [3]:
%cd phase1
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 feature_engineering.ipynb
%cd ..
!ls -lh data/processed

/content/smart_retail/phase1
[NbConvertApp] Converting notebook feature_engineering.ipynb to notebook
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
[NbConvertApp] Writing 39091 bytes to feature_engineering.ipynb
/content/smart_retail
total 120M
-rw-r--r-- 1 root root  20M Nov 28 11:25 cleaned_merged.parquet
-rw-r--r-- 1 root root 101M Nov 28 11:26 features_engineered.parquet
-rw-r--r-- 1 root root 6.8K Nov 28 11:25 README.md


Data load and Split


In [4]:
import numpy as np, pandas as pd, json
df = pd.read_parquet("data/processed/features_engineered.parquet")
df = df[(df["reliable_period"]==1) & (df["is_store_open"]==1)].copy()
df["y"] = np.log1p(df["sales"])

val_start = df["date"].max() - pd.Timedelta(days=27)
train = df[df["date"] < val_start].copy()
valid = df[df["date"] >= val_start].copy()
valid["week"] = ((valid["date"] - val_start).dt.days // 7) + 1
drop = ["id","date","sales","y","week","city","state","store_type","family",
        "is_store_open","reliable_period","promo_recorded"]
features = [c for c in df.columns if c not in drop and train[c].nunique() > 1]
print(len(features), train.shape, valid.shape)

52 (1280631, 63) (49896, 64)


XGBoost train

In [5]:
import xgboost as xgb, os
os.makedirs("models", exist_ok=True)
xgb_model = xgb.XGBRegressor(n_estimators=502, max_depth=10, learning_rate=0.03,
    subsample=1.0, colsample_bytree=0.8, min_child_weight=1,
    tree_method="hist", random_state=42, n_jobs=-1)
xgb_model.fit(train[features], train["y"])
xgb_model.save_model("models/xgb_model.json")
json.dump(features, open("models/xgb_features.json","w"))

def metrics(y_log, p_log, sales):
    p_log = np.clip(p_log, 0, None); p = np.expm1(p_log); nz = sales > 0
    return {"RMSLE": np.sqrt(np.mean((p_log-y_log)**2)),
            "RMSE": np.sqrt(np.mean((p-sales)**2)),
            "MAPE": np.mean(np.abs(p[nz]-sales[nz])/sales[nz])*100,
            "R2": 1-np.sum((sales-p)**2)/np.sum((sales-sales.mean())**2)}
def weekly_mean(vdf, p_log):
    rows=[]
    for w in sorted(vdf["week"].unique()):
        m=(vdf["week"]==w).values
        rows.append(metrics(vdf["y"].values[m], p_log[m], vdf["sales"].values[m]))
    return pd.DataFrame(rows).mean()

rep_x = weekly_mean(valid, xgb_model.predict(valid[features]))
print(rep_x.round(4))

RMSLE      0.3971
RMSE     214.9129
MAPE      31.9118
R2         0.9707
dtype: float64


GD + L1 train


In [6]:
from sklearn.preprocessing import StandardScaler

class GDRegression:
    def __init__(self, lr=0.05, epochs=500, l1=0.0, l2=0.0):
        self.lr, self.epochs, self.l1, self.l2 = lr, epochs, l1, l2
    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d, dtype=np.float32); self.b = np.float32(0)
        for _ in range(self.epochs):
            err = X @ self.w + self.b - y
            dw = X.T @ err / n + 2*self.l2*self.w + self.l1*np.sign(self.w)
            self.w -= self.lr * dw
            self.b -= self.lr * err.mean()
        return self
    def predict(self, X):
        return X @ self.w + self.b

gd_train = train.dropna(subset=features).copy()
gd_valid = valid.dropna(subset=features).copy()
cat_cols = ["store_nbr","family_code","store_type_code","store_cluster","day_of_week","month"]
skip = ["city_code","state_code","quarter","onpromotion"]
log_cols = ["sales_lag_7d","sales_lag_14d","sales_lag_21d","sales_lag_28d",
            "sales_same_dow_mean_4w","rolling_mean_7d","rolling_std_7d",
            "rolling_mean_14d","rolling_std_14d","rolling_mean_28d","rolling_std_28d",
            "onpromotion_lag_7d","transactions_lag_7d","transactions_roll_7d"]
num_cols = [c for c in features if c not in cat_cols + skip]
cap = gd_train["promo_intensity_vs_28d"].quantile(0.99)

def prep(d):
    X = d[num_cols + cat_cols].copy()
    for c in log_cols:
        if c in X: X[c] = np.log1p(X[c].clip(lower=0))
    X["promo_intensity_vs_28d"] = np.log1p(X["promo_intensity_vs_28d"].clip(0, cap))
    return X

Xtr, Xva = prep(gd_train), prep(gd_valid)
n_tr = len(Xtr)
allX = pd.get_dummies(pd.concat([Xtr, Xva]), columns=cat_cols,
                      drop_first=True, dtype=np.float32).astype(np.float32)
scaler = StandardScaler().fit(allX.iloc[:n_tr])
Xtr_s = scaler.transform(allX.iloc[:n_tr]).astype(np.float32)
Xva_s = scaler.transform(allX.iloc[n_tr:]).astype(np.float32)

gd = GDRegression(lr=0.05, epochs=500, l1=0.01).fit(Xtr_s, gd_train["y"].values.astype(np.float32))
np.savez("models/gd_l1_model.npz", w=gd.w, b=gd.b, mean=scaler.mean_, scale=scaler.scale_,
         feature_names=np.array(allX.columns), promo_cap=cap)
rep_g = weekly_mean(gd_valid, gd.predict(Xva_s))
print(rep_g.round(4))

pd.DataFrame({"XGBoost": rep_x, "GD + L1 (lambda=0.01)": rep_g}).T.round(4).to_csv("models/validation_metrics.csv")

RMSLE      0.4459
RMSE     261.0714
MAPE      35.7618
R2         0.9569
dtype: float64


In [7]:
!zip -qr models.zip models
from google.colab import files
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>